In [5]:
# Creates a connection object that pandas can use to talk to a SQL database.
from dotenv import load_dotenv
import os
from sqlalchemy import create_engine
import pandas as pd

load_dotenv()
engine = create_engine(os.getenv('SUPABASE_URL'))

In [7]:

# multi joins 
query_1 = """
SELECT 
    o.order_id,
    o.order_purchase_timestamp,
    c.customer_city,
    c.customer_state,
    ct.product_category_name_english AS category,
    s.seller_city,
    s.seller_state,
    oi.price,
    oi.freight_value
FROM orders o
JOIN order_items oi  ON o.order_id = oi.order_id
JOIN customers c     ON o.customer_id = c.customer_id
JOIN products p      ON oi.product_id = p.product_id
JOIN category_translation ct 
                     ON p.product_category_name = ct.product_category_name
JOIN sellers s       ON oi.seller_id = s.seller_id
LIMIT 20;
"""

df1 = pd.read_sql(query_1, engine)
df1

,order_id,order_purchase_timestamp,customer_city,customer_state,category,seller_city,seller_state,price,freight_value
0,000229ec398224ef6ca0657da4fc703e,2018-01-14 14:33:31,para de minas,MG,furniture_decor,borda da mata,MG,199.00,17.87
1,00061f2a7bc09da83e415a52dc8a4af1,2018-03-24 22:16:10,piracicaba,SP,health_beauty,santo andre,SP,59.99,8.88
2,001b76dd48a5b1eee3e87778daa40df8,2018-03-26 17:42:53,santa salete,SP,computers_accessories,sao jose dos campos,SP,81.99,13.01
3,003324c70b19a16798817b2b3640e721,2017-05-18 19:04:48,diamantina,MG,luggage_accessories,borda da mata,MG,102.90,14.45
4,003324c70b19a16798817b2b3640e721,2017-05-18 19:04:48,diamantina,MG,luggage_accessories,borda da mata,MG,102.90,14.45
5,003f201cdd39cdd59b6447cff2195456,2018-02-21 16:59:24,rio de janeiro,RJ,computers_accessories,maringa,PR,85.00,17.03
6,003f201cdd39cdd59b6447cff2195456,2018-02-21 16:59:24,rio de janeiro,RJ,computers_accessories,maringa,PR,85.00,17.03
7,0045e3085f083f0f38d24bb3f22e6593,2017-08-10 16:13:36,sao jose dos campos,SP,health_beauty,ribeirao preto,SP,116.90,13.84
8,00588f5f20fee630b83d4709e0a42715,2017-08-18 16:02:24,ipixuna do para,PA,telephony,jacarei,SP,89.00,25.90
9,0064d74f7de7917d54768d7722ad56ac,2017-09-10 21:45:29,niteroi,RJ,cool_stuff,sao paulo,SP,189.90,31.17


In [8]:

#windows
query_2a = """
WITH customer_spend AS (
    SELECT 
        c.customer_unique_id,
        c.customer_city,
        c.customer_state,
        SUM(oi.price + oi.freight_value) AS total_spend,
        COUNT(DISTINCT o.order_id) AS total_orders
    FROM customers c
    JOIN orders o       ON c.customer_id = o.customer_id
    JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY c.customer_unique_id, c.customer_city, c.customer_state
)
SELECT 
    customer_unique_id,
    customer_city,
    customer_state,
    total_spend,
    total_orders,
    RANK() OVER (ORDER BY total_spend DESC) AS spend_rank,
    DENSE_RANK() OVER (ORDER BY total_spend DESC) AS spend_dense_rank
FROM customer_spend
LIMIT 20;
"""

df2a = pd.read_sql(query_2a, engine)
df2a

,customer_unique_id,customer_city,customer_state,total_spend,total_orders,spend_rank,spend_dense_rank
0,0a0a92112bd4c708ca5fde585afaa872,rio de janeiro,RJ,13664.08,1,1,1
1,da122df9eeddfedc1dc1f5349a1a690c,araruama,RJ,7571.63,2,2,2
2,763c8b1c9c68a0229c42c9fc6f662b93,vila velha,ES,7274.88,1,3,3
3,dc4802a71eae9be1dd28f5d788ceb526,campo grande,MS,6929.31,1,4,4
4,459bef486812aa25204be022145caa62,vitoria,ES,6922.21,1,5,5
5,ff4159b92c40ebe40454e3e6a7c35ed6,marilia,SP,6726.66,1,6,6
6,4007669dec559734d6f53e029e360987,divinopolis,MG,6081.54,1,7,7
7,5d0a2980b292d049061542014e8960bf,goiania,GO,4809.44,1,8,8
8,eebb5dda148d3893cdaf5b5ca3040ccb,maua,SP,4764.34,1,9,9
9,48e1ac109decbb87765a3eade6854098,joao pessoa,PB,4681.78,1,10,10


In [9]:


query_2b = """
SELECT 
    DATE(o.order_purchase_timestamp) AS order_date,
    SUM(oi.price) AS daily_revenue,
    SUM(SUM(oi.price)) OVER (ORDER BY DATE(o.order_purchase_timestamp)) AS running_total
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
GROUP BY DATE(o.order_purchase_timestamp)
ORDER BY order_date
LIMIT 30;
"""

df2b = pd.read_sql(query_2b, engine)
df2b

,order_date,daily_revenue,running_total
0,2016-09-04,72.89,72.89
1,2016-09-05,59.50,132.39
2,2016-09-15,134.97,267.36
3,2016-10-02,100.00,367.36
4,2016-10-03,463.48,830.84
5,2016-10-04,9940.96,10771.80
6,2016-10-05,8343.25,19115.05
7,2016-10-06,7960.51,27075.56
8,2016-10-07,7228.05,34303.61
9,2016-10-08,8441.85,42745.46


In [13]:
query_3 = """
WITH monthly_revenue AS (
    SELECT 
        DATE_TRUNC('month', o.order_purchase_timestamp::timestamp) AS month,
        COUNT(DISTINCT o.order_id) AS total_orders,
        SUM(oi.price) AS revenue,
        SUM(oi.freight_value) AS total_freight
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY DATE_TRUNC('month', o.order_purchase_timestamp::timestamp)
)
SELECT 
    month,
    total_orders,
    ROUND(revenue::numeric, 2) AS revenue,
    ROUND(total_freight::numeric, 2) AS total_freight,
    LAG(revenue) OVER (ORDER BY month) AS prev_month_revenue,
    ROUND(
        ((revenue - LAG(revenue) OVER (ORDER BY month)) 
        / NULLIF(LAG(revenue) OVER (ORDER BY month), 0) * 100)::numeric,
        1
    ) AS mom_growth_pct
      
FROM monthly_revenue
ORDER BY month;
"""

df3 = pd.read_sql(query_3, engine)
df3

,month,total_orders,revenue,total_freight,prev_month_revenue,mom_growth_pct
0,2016-09-01,1,134.97,8.49,NaN,NaN
1,2016-10-01,265,40325.11,6165.55,134.97,29777.1
2,2016-12-01,1,10.90,8.72,40325.11,-100.0
3,2017-01-01,750,111798.36,15684.01,10.90,1025573.0
4,2017-02-01,1653,234223.40,37015.92,111798.36,109.5
5,2017-03-01,2546,359198.85,55132.10,234223.40,53.4
6,2017-04-01,2303,340669.68,50142.72,359198.85,-5.2
7,2017-05-01,3546,489338.25,77513.15,340669.68,43.6
8,2017-06-01,3135,421923.37,68127.00,489338.25,-13.8
9,2017-07-01,3872,481604.52,84694.56,421923.37,14.1


In [11]:
query_4 = """
WITH seller_revenue AS (
    SELECT 
        s.seller_id,
        s.seller_city,
        s.seller_state,
        COUNT(DISTINCT o.order_id) AS total_orders,
        SUM(oi.price) AS total_revenue,
        ROUND(AVG(oi.price)::numeric, 2) AS avg_order_value
    FROM sellers s
    JOIN order_items oi ON s.seller_id = oi.seller_id
    JOIN orders o       ON oi.order_id = o.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY s.seller_id, s.seller_city, s.seller_state
),
seller_reviews AS (
    SELECT 
        oi.seller_id,
        ROUND(AVG(r.review_score)::numeric, 2) AS avg_review_score,
        COUNT(r.review_id) AS total_reviews
    FROM order_items oi
    JOIN reviews r ON oi.order_id = r.order_id
    GROUP BY oi.seller_id
)
SELECT 
    sr.seller_id,
    sr.seller_city,
    sr.seller_state,
    sr.total_orders,
    ROUND(sr.total_revenue::numeric, 2) AS total_revenue,
    sr.avg_order_value,
    COALESCE(rv.avg_review_score, 0) AS avg_review_score,
    COALESCE(rv.total_reviews, 0) AS total_reviews
FROM seller_revenue sr
LEFT JOIN seller_reviews rv ON sr.seller_id = rv.seller_id
ORDER BY sr.total_revenue DESC
LIMIT 10;
"""

df4 = pd.read_sql(query_4, engine)
df4

,seller_id,seller_city,seller_state,total_orders,total_revenue,avg_order_value,avg_review_score,total_reviews
0,4869f7a5dfa277a7dca6462dcf3b52b2,guariba,SP,1124,226987.93,197.72,4.12,1148
1,53243585a1d6dc2643021fd1853d8905,lauro de freitas,BA,348,217940.44,544.85,4.08,408
2,4a3ca9315b744ce9f8e9374361493884,ibitinga,SP,1772,196882.12,101.02,3.80,1984
3,fa1c13f2614d7b5c4749cbc52fecda94,sumare,SP,578,190917.14,329.74,4.34,582
4,7c67e1448b00f6e969d365cea6b010ab,itaquaquecetuba,SP,973,186570.05,137.69,3.35,1367
5,7e93a43ef30c4f03f38b393420bc753a,barueri,SP,319,165981.49,515.47,4.21,339
6,da8622b14eb17ae2831f4ac5b9dab84a,piracicaba,SP,1311,159816.87,103.24,4.07,1568
7,7a67c85e85bb2ce8582c35f2203ad736,sao paulo,SP,1145,139658.69,120.92,4.23,1166
8,1025f0e2d44d7041d6cf58b6550e0bfa,sao paulo,SP,910,138208.56,97.33,3.85,1431
9,955fee9216a65b617aa5c0531780ce60,sao paulo,SP,1261,131836.71,89.56,4.05,1489
